In [9]:
# %matplotlib notebook

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager, rc

# 한글깨짐처리
font_loc = "c:/Windows/Fonts/malgun.ttf"
font_name=font_manager.FontProperties(fname=font_loc).get_name()
rc('font', family=font_name)

In [10]:
import requests
import pandas as pd
import time

# 1) 카카오 API 설정
url = "https://dapi.kakao.com/v2/local/search/keyword.json"
headers = {"Authorization": "KakaoAK YOUR_KAKAO_API_KEY"}

# 2) 함수
def get_school_location(query):
    try:
        params = {'query': query}
        res = requests.get(url, headers=headers, params=params, timeout=5)
        a = res.json()

        docs = a.get('documents', [])

        if not docs:
            return pd.Series([None, None, None, None])

        first = docs[0]

        place_name = first.get('place_name', None)
        road_addr = first.get('road_address_name', None)
        lat = first.get('y', None)   # 위도
        lon = first.get('x', None)   # 경도

        time.sleep(0.15)  # 너무 빠른 호출 방지
        return pd.Series([place_name, road_addr, lat, lon])

    except Exception as e:
        print(f"검색 실패: {query} / 에러: {e}")
        return pd.Series([None, None, None, None])

# 3) 기존 csv 읽기
df = pd.read_csv("돌봄교실.csv", encoding="cp949")

# 4) 검색어 만들기
# 컬럼명이 정확히 '시군구', '학교명'인지 확인
df['검색어'] = df['시군구'].astype(str).str.strip() + " " + df['학교명'].astype(str).str.strip()

# 5) 함수 적용해서 새 컬럼 붙이기
df[['카카오학교명', '도로명주소', '위도', '경도']] = df['검색어'].apply(get_school_location)

# 6) 필요 없으면 검색어 컬럼 삭제
# df = df.drop(columns=['검색어'])

# 7) 새 csv로 저장
df.to_csv("돌봄교실_주소좌표추가.csv", index=False, encoding="utf-8-sig")

print("완료")

완료


In [12]:
df = pd.read_csv('돌봄교실_주소좌표추가.csv', encoding='utf-8-sig')
df

,시도,시군구,학교명,Unnamed: 3,검색어,카카오학교명,도로명주소,위도,경도
0,서울시교육청,동대문구,군자초,NaN,동대문구 군자초,서울군자초등학교,서울 동대문구 한천로6길 21,37.564430,127.061646
1,서울시교육청,동대문구,답십리초,NaN,동대문구 답십리초,서울답십리초등학교,서울 동대문구 전농로3길 23,37.568725,127.055474
2,서울시교육청,동대문구,동답초,NaN,동대문구 동답초,서울동답초등학교,서울 동대문구 답십리로60길 12,37.571910,127.064387
3,서울시교육청,중랑구,동원초,NaN,중랑구 동원초,서울동원초등학교,서울 중랑구 송림길 114,37.604724,127.105379
4,서울시교육청,중랑구,망우초,NaN,중랑구 망우초,서울망우초등학교,서울 중랑구 망우로72길 49,37.598516,127.103301
...,...,...,...,...,...,...,...,...,...
1620,NaN,NaN,NaN,NaN,nan nan,엔에이엔(NAN),경기 의정부시 녹양로 87,37.758587,127.034046
1621,NaN,NaN,NaN,NaN,nan nan,엔에이엔(NAN),경기 의정부시 녹양로 87,37.758587,127.034046
1622,NaN,NaN,NaN,NaN,nan nan,엔에이엔(NAN),경기 의정부시 녹양로 87,37.758587,127.034046
1623,NaN,NaN,NaN,NaN,nan nan,엔에이엔(NAN),경기 의정부시 녹양로 87,37.758587,127.034046


In [14]:
df = pd.read_csv('돌봄교실_주소좌표추가.csv', usecols=[0,1,5,6,7,8], encoding='utf-8-sig')
df

,시도,시군구,카카오학교명,도로명주소,위도,경도
0,서울시교육청,동대문구,서울군자초등학교,서울 동대문구 한천로6길 21,37.564430,127.061646
1,서울시교육청,동대문구,서울답십리초등학교,서울 동대문구 전농로3길 23,37.568725,127.055474
2,서울시교육청,동대문구,서울동답초등학교,서울 동대문구 답십리로60길 12,37.571910,127.064387
3,서울시교육청,중랑구,서울동원초등학교,서울 중랑구 송림길 114,37.604724,127.105379
4,서울시교육청,중랑구,서울망우초등학교,서울 중랑구 망우로72길 49,37.598516,127.103301
...,...,...,...,...,...,...
1620,NaN,NaN,엔에이엔(NAN),경기 의정부시 녹양로 87,37.758587,127.034046
1621,NaN,NaN,엔에이엔(NAN),경기 의정부시 녹양로 87,37.758587,127.034046
1622,NaN,NaN,엔에이엔(NAN),경기 의정부시 녹양로 87,37.758587,127.034046
1623,NaN,NaN,엔에이엔(NAN),경기 의정부시 녹양로 87,37.758587,127.034046


In [15]:
df.to_csv("돌봄교실_주소좌표추가.csv", index=False, encoding='utf-8-sig')

In [7]:
df = pd.read_csv('어린이집합계2.csv', encoding='cp949')
df.head()

,행정동,어린이집수,유아인구합계,시군구
0,가락1동,14,1520,NaN
1,가락2동,14,868,NaN
2,가락본동,11,705,NaN
3,가리봉동,2,94,NaN
4,가산동,9,364,NaN


In [11]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

def count_within(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    return int((df['거리'] <= radius).sum())

def normalize(value, v_min, v_max):
    value = max(v_min, min(v_max, value))
    return round((value - v_min) / (v_max - v_min), 4)

# ============================================================
# 반경 기준
# ============================================================
ELEM_RADIUS        = 1000
MIDDLE_RADIUS      = 1500
ACADEMY_RADIUS     = 500
AFTERSCHOOL_RADIUS = 1000

# ============================================================
# 정규화 기준값 (실제 데이터 분포 확인 후 조정 필요)
# ============================================================
ELEM_DIST_MAX   = 0.003
ELEM_CNT_MAX    = 3
MIDDLE_DIST_MAX = 0.002
MIDDLE_CNT_MAX  = 2
ACADEMY_MAX     = 20
AFTERSCHOOL_MAX = 3

# ============================================================
# 데이터 로드
# ============================================================
df_elem        = pd.read_csv('교육 - 초등학교.csv',    encoding='cp949')
df_middle      = pd.read_csv('교육 - 중학교.csv',      encoding='cp949')
df_academy     = pd.read_csv('교육 - 학원.csv',        encoding='cp949')
df_afterschool = pd.read_csv('최종-방과후돌봄교실.csv', encoding='cp949')

# 위도/경도 숫자로 변환
for df in [df_elem, df_middle, df_afterschool]:
    df['위도'] = pd.to_numeric(df['위도'], errors='coerce')
    df['경도'] = pd.to_numeric(df['경도'], errors='coerce')

df_academy['위도'] = pd.to_numeric(df_academy['위도'], errors='coerce')
df_academy['경도'] = pd.to_numeric(df_academy['경도'], errors='coerce')

# ============================================================
# 주소 입력
# ============================================================
address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
    # 점수 계산
    s_elem_dist   = get_access_score(my_lat, my_lng, df_elem,        '위도', '경도', ELEM_RADIUS)
    n_elem        = count_within    (my_lat, my_lng, df_elem,        '위도', '경도', ELEM_RADIUS)
    s_middle_dist = get_access_score(my_lat, my_lng, df_middle,      '위도', '경도', MIDDLE_RADIUS)
    n_middle      = count_within    (my_lat, my_lng, df_middle,      '위도', '경도', MIDDLE_RADIUS)
    n_academy     = count_within    (my_lat, my_lng, df_academy,     '위도', '경도', ACADEMY_RADIUS)
    n_afterschool = count_within    (my_lat, my_lng, df_afterschool, '위도', '경도', AFTERSCHOOL_RADIUS)

    # 정규화 (0~1)
    v1 = normalize(s_elem_dist,   0, ELEM_DIST_MAX)
    v2 = normalize(n_elem,        0, ELEM_CNT_MAX)
    v3 = normalize(s_middle_dist, 0, MIDDLE_DIST_MAX)
    v4 = normalize(n_middle,      0, MIDDLE_CNT_MAX)
    v5 = normalize(n_academy,     0, ACADEMY_MAX)
    v6 = normalize(n_afterschool, 0, AFTERSCHOOL_MAX)

    # 100점 만점 환산
    score = round((v1 + v2 + v3 + v4 + v5 + v6) / 6 * 100, 2)

    # 결과 출력
    print(f"\n📍 {address}")
    print(f"   좌표: ({my_lat}, {my_lng})")
    print()
    print(f"초등학교 접근성 ({ELEM_RADIUS}m):        {s_elem_dist:.4f}  → V1: {v1:.4f}")
    print(f"초등학교 개수 ({ELEM_RADIUS}m):          {n_elem}개  → V2: {v2:.4f}")
    print(f"중학교 접근성 ({MIDDLE_RADIUS}m):       {s_middle_dist:.4f}  → V3: {v3:.4f}")
    print(f"중학교 개수 ({MIDDLE_RADIUS}m):         {n_middle}개  → V4: {v4:.4f}")
    print(f"학원 밀집도 ({ACADEMY_RADIUS}m):         {n_academy}개  → V5: {v5:.4f}")
    print(f"돌봄교실 운영 학교 ({AFTERSCHOOL_RADIUS}m):  {n_afterschool}개  → V6: {v6:.4f}")
    print()
    print(f"※ 어린이집 데이터 미포함 임시 버전")
    print(f"교육점수 = (V1+V2+V3+V4+V5+V6)/6 × 100")
    print(f"         = {score}점")

주소를 입력하세요:  서울시 도봉구 도봉로110마길 42



📍 서울시 도봉구 도봉로110마길 42
   좌표: (37.6445244741978, 127.036204968923)

초등학교 접근성 (1000m):        0.0042  → V1: 1.0000
초등학교 개수 (1000m):          3개  → V2: 1.0000
중학교 접근성 (1500m):       0.0090  → V3: 1.0000
중학교 개수 (1500m):         8개  → V4: 1.0000
학원 밀집도 (500m):         42개  → V5: 1.0000
돌봄교실 운영 학교 (1000m):  3개  → V6: 1.0000

※ 어린이집 데이터 미포함 임시 버전
교육점수 = (V1+V2+V3+V4+V5+V6)/6 × 100
         = 100.0점


In [12]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1)
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

def count_within(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1)
    return int((df['거리'] <= radius).sum())

ELEM_RADIUS        = 1000
MIDDLE_RADIUS      = 1500
ACADEMY_RADIUS     = 500
AFTERSCHOOL_RADIUS = 1000

df_elem        = pd.read_csv('교육 - 초등학교.csv',    encoding='cp949')
df_middle      = pd.read_csv('교육 - 중학교.csv',      encoding='cp949')
df_academy     = pd.read_csv('교육 - 학원.csv',        encoding='cp949')
df_afterschool = pd.read_csv('최종-방과후돌봄교실.csv', encoding='cp949')

for df in [df_elem, df_middle, df_afterschool]:
    df['위도'] = pd.to_numeric(df['위도'], errors='coerce')
    df['경도'] = pd.to_numeric(df['경도'], errors='coerce')
df_academy['위도'] = pd.to_numeric(df_academy['위도'], errors='coerce')
df_academy['경도'] = pd.to_numeric(df_academy['경도'], errors='coerce')

# 수도권 다양한 지역 샘플 (도심/외곽/위성도시 골고루)
test_addresses = [
    "서울특별시 강남구 테헤란로 152",
    "서울특별시 노원구 동일로 1292",
    "서울특별시 도봉구 도봉로110마길 42",
    "서울특별시 은평구 통일로 684",
    "서울특별시 중랑구 망우로 353",
    "서울특별시 강서구 화곡로 302",
    "경기도 성남시 분당구 정자일로 95",
    "경기도 고양시 일산동구 중앙로 1256",
    "경기도 남양주시 화도읍 마석로 15",
    "경기도 김포시 사우동 김포한강로 301",
    "경기도 용인시 기흥구 중부대로 579",
    "인천광역시 연수구 인천타워대로 323",
    "인천광역시 부평구 부평대로 168",
    "경기도 하남시 대청로 10",
]

results = []
for addr in test_addresses:
    lat, lng = get_coord(addr)
    if lat:
        results.append({
            '주소':           addr,
            's_elem_dist':   round(get_access_score(lat, lng, df_elem,        '위도', '경도', ELEM_RADIUS), 4),
            'n_elem':        count_within(lat, lng, df_elem,        '위도', '경도', ELEM_RADIUS),
            's_middle_dist': round(get_access_score(lat, lng, df_middle,      '위도', '경도', MIDDLE_RADIUS), 4),
            'n_middle':      count_within(lat, lng, df_middle,      '위도', '경도', MIDDLE_RADIUS),
            'n_academy':     count_within(lat, lng, df_academy,     '위도', '경도', ACADEMY_RADIUS),
            'n_afterschool': count_within(lat, lng, df_afterschool, '위도', '경도', AFTERSCHOOL_RADIUS),
        })
        print(f"완료: {addr}")

df_result = pd.DataFrame(results)

print("\n===== 전체 결과 =====")
print(df_result.to_string(index=False))

print("\n===== 분포 (describe) =====")
print(df_result.drop(columns='주소').describe().round(4).to_string())

print("\n===== 상위 10% (90th percentile) → MAX값 후보 =====")
cols = ['s_elem_dist', 'n_elem', 's_middle_dist', 'n_middle', 'n_academy', 'n_afterschool']
for col in cols:
    p90 = df_result[col].quantile(0.9)
    print(f"  {col}: {p90:.4f}")

완료: 서울특별시 강남구 테헤란로 152
완료: 서울특별시 도봉구 도봉로110마길 42
완료: 서울특별시 은평구 통일로 684
완료: 서울특별시 중랑구 망우로 353
완료: 서울특별시 강서구 화곡로 302
완료: 경기도 성남시 분당구 정자일로 95
완료: 경기도 고양시 일산동구 중앙로 1256
완료: 경기도 용인시 기흥구 중부대로 579
완료: 인천광역시 연수구 인천타워대로 323
완료: 인천광역시 부평구 부평대로 168
완료: 경기도 하남시 대청로 10

===== 전체 결과 =====
                   주소  s_elem_dist  n_elem  s_middle_dist  n_middle  n_academy  n_afterschool
   서울특별시 강남구 테헤란로 152       0.0012       1         0.0051         6         52              1
서울특별시 도봉구 도봉로110마길 42       0.0042       3         0.0090         8         42              3
    서울특별시 은평구 통일로 684       0.0086       5         0.0000         0         37              5
    서울특별시 중랑구 망우로 353       0.0067       5         0.0090         7         32              5
    서울특별시 강서구 화곡로 302       0.0083       5         0.0065         7         21              4
  경기도 성남시 분당구 정자일로 95       0.0161       3         0.0124         6         74              3
경기도 고양시 일산동구 중앙로 1256       0.0026       2         0.0044         

In [13]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

def count_within(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    return int((df['거리'] <= radius).sum())

def normalize(value, v_min, v_max):
    value = max(v_min, min(v_max, value))
    return round((value - v_min) / (v_max - v_min), 4)

# ============================================================
# 반경 기준
# ============================================================
# 초등학교 1000m : 교육환경 보호에 관한 법률 - 초등학교 통학거리 법적 기준
# 중학교   1500m : 초등학교 법적 기준 준용, 중학생 보행 능력 고려 상향
# 학원      500m : 어린이보호구역 지정 기준 - 학교 정문 반경 500m
# 어린이집 2000m : 국무조정실 생활SOC 3개년계획 - 국공립어린이집 최저설치 기준
# 돌봄교실 1000m : 초등학교 내 운영이므로 초등학교 통학거리 기준 동일 적용

ELEM_RADIUS        = 1000
MIDDLE_RADIUS      = 1500
ACADEMY_RADIUS     = 500
AFTERSCHOOL_RADIUS = 1000

# ============================================================
# 정규화 기준값 - 수도권 샘플 지역 90th percentile 기준
# ============================================================
ELEM_DIST_MAX   = 0.0115
ELEM_CNT_MAX    = 5
MIDDLE_DIST_MAX = 0.0090
MIDDLE_CNT_MAX  = 7
ACADEMY_MAX     = 52
AFTERSCHOOL_MAX = 5

# ============================================================
# 데이터 로드
# ============================================================
df_elem        = pd.read_csv('교육 - 초등학교.csv',    encoding='cp949')
df_middle      = pd.read_csv('교육 - 중학교.csv',      encoding='cp949')
df_academy     = pd.read_csv('교육 - 학원.csv',        encoding='cp949')
df_afterschool = pd.read_csv('최종-방과후돌봄교실.csv', encoding='cp949')

# 위도/경도 숫자로 변환
for df in [df_elem, df_middle, df_afterschool]:
    df['위도'] = pd.to_numeric(df['위도'], errors='coerce')
    df['경도'] = pd.to_numeric(df['경도'], errors='coerce')

df_academy['위도'] = pd.to_numeric(df_academy['위도'], errors='coerce')
df_academy['경도'] = pd.to_numeric(df_academy['경도'], errors='coerce')

# ============================================================
# 주소 입력
# ============================================================
address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
    # 점수 계산
    s_elem_dist   = get_access_score(my_lat, my_lng, df_elem,        '위도', '경도', ELEM_RADIUS)
    n_elem        = count_within    (my_lat, my_lng, df_elem,        '위도', '경도', ELEM_RADIUS)
    s_middle_dist = get_access_score(my_lat, my_lng, df_middle,      '위도', '경도', MIDDLE_RADIUS)
    n_middle      = count_within    (my_lat, my_lng, df_middle,      '위도', '경도', MIDDLE_RADIUS)
    n_academy     = count_within    (my_lat, my_lng, df_academy,     '위도', '경도', ACADEMY_RADIUS)
    n_afterschool = count_within    (my_lat, my_lng, df_afterschool, '위도', '경도', AFTERSCHOOL_RADIUS)

    # 정규화 (0~1)
    v1 = normalize(s_elem_dist,   0, ELEM_DIST_MAX)
    v2 = normalize(n_elem,        0, ELEM_CNT_MAX)
    v3 = normalize(s_middle_dist, 0, MIDDLE_DIST_MAX)
    v4 = normalize(n_middle,      0, MIDDLE_CNT_MAX)
    v5 = normalize(n_academy,     0, ACADEMY_MAX)
    v6 = normalize(n_afterschool, 0, AFTERSCHOOL_MAX)

    # 100점 만점 환산
    score = round((v1 + v2 + v3 + v4 + v5 + v6) / 6 * 100, 2)

    # 결과 출력
    print(f"\n📍 {address}")
    print(f"   좌표: ({my_lat}, {my_lng})")
    print()
    print(f"초등학교 접근성 ({ELEM_RADIUS}m):        {s_elem_dist:.4f}  → V1: {v1:.4f}")
    print(f"초등학교 개수 ({ELEM_RADIUS}m):          {n_elem}개  → V2: {v2:.4f}")
    print(f"중학교 접근성 ({MIDDLE_RADIUS}m):       {s_middle_dist:.4f}  → V3: {v3:.4f}")
    print(f"중학교 개수 ({MIDDLE_RADIUS}m):         {n_middle}개  → V4: {v4:.4f}")
    print(f"학원 밀집도 ({ACADEMY_RADIUS}m):         {n_academy}개  → V5: {v5:.4f}")
    print(f"돌봄교실 운영 학교 ({AFTERSCHOOL_RADIUS}m):  {n_afterschool}개  → V6: {v6:.4f}")
    print()
    print(f"※ 어린이집 데이터 미포함 임시 버전")
    print(f"교육점수 = (V1+V2+V3+V4+V5+V6)/6 × 100")
    print(f"         = {score}점")

주소를 입력하세요:  서울시 도봉구 도봉로110마길 42



📍 서울시 도봉구 도봉로110마길 42
   좌표: (37.6445244741978, 127.036204968923)

초등학교 접근성 (1000m):        0.0042  → V1: 0.3676
초등학교 개수 (1000m):          3개  → V2: 0.6000
중학교 접근성 (1500m):       0.0090  → V3: 1.0000
중학교 개수 (1500m):         8개  → V4: 1.0000
학원 밀집도 (500m):         42개  → V5: 0.8077
돌봄교실 운영 학교 (1000m):  3개  → V6: 0.6000

※ 어린이집 데이터 미포함 임시 버전
교육점수 = (V1+V2+V3+V4+V5+V6)/6 × 100
         = 72.92점


In [20]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1)
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

def count_within(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1)
    return int((df['거리'] <= radius).sum())

ELEM_RADIUS        = 1000
MIDDLE_RADIUS      = 1500
ACADEMY_RADIUS     = 500
DAYCARE_RADIUS     = 500   # 어린이집
AFTERSCHOOL_RADIUS = 1000

# ============================================================
# 데이터 로드
# ============================================================
df_elem        = pd.read_csv('교육 - 초등학교.csv',    encoding='cp949')
df_middle      = pd.read_csv('교육 - 중학교.csv',      encoding='cp949')
df_academy     = pd.read_csv('교육 - 학원.csv',        encoding='cp949')
df_afterschool = pd.read_csv('최종-방과후돌봄교실.csv', encoding='cp949')
df_daycare     = pd.read_csv('어린이집정제.csv', encoding='cp949')

# 위도/경도 숫자로 변환
for df in [df_elem, df_middle, df_afterschool, df_daycare]:
    df['위도'] = pd.to_numeric(df['위도'], errors='coerce')
    df['경도'] = pd.to_numeric(df['경도'], errors='coerce')
df_academy['위도'] = pd.to_numeric(df_academy['위도'], errors='coerce')
df_academy['경도'] = pd.to_numeric(df_academy['경도'], errors='coerce')

# ============================================================
# 수도권 샘플 주소
# ============================================================
test_addresses = [
    "서울특별시 강남구 테헤란로 152",
    "서울특별시 노원구 동일로 1292",
    "서울특별시 도봉구 도봉로110마길 42",
    "서울특별시 은평구 통일로 684",
    "서울특별시 중랑구 망우로 353",
    "서울특별시 강서구 화곡로 302",
    "경기도 성남시 분당구 정자일로 95",
    "경기도 고양시 일산동구 중앙로 1256",
    "경기도 남양주시 화도읍 마석로 15",
    "경기도 김포시 사우동 김포한강로 301",
    "경기도 용인시 기흥구 중부대로 579",
    "인천광역시 연수구 인천타워대로 323",
    "인천광역시 부평구 부평대로 168",
    "경기도 하남시 대청로 10",
]

results = []
for addr in test_addresses:
    lat, lng = get_coord(addr)
    if lat:
        results.append({
            '주소':           addr,
            's_elem_dist':   round(get_access_score(lat, lng, df_elem,        '위도', '경도', ELEM_RADIUS), 4),
            'n_elem':        count_within(lat, lng, df_elem,        '위도', '경도', ELEM_RADIUS),
            's_middle_dist': round(get_access_score(lat, lng, df_middle,      '위도', '경도', MIDDLE_RADIUS), 4),
            'n_middle':      count_within(lat, lng, df_middle,      '위도', '경도', MIDDLE_RADIUS),
            'n_academy':     count_within(lat, lng, df_academy,     '위도', '경도', ACADEMY_RADIUS),
            'n_daycare': count_within(lat, lng, df_daycare, '위도', '경도', DAYCARE_RADIUS),
            'n_afterschool': count_within(lat, lng, df_afterschool, '위도', '경도', AFTERSCHOOL_RADIUS),
        })
        print(f"완료: {addr}")

df_result = pd.DataFrame(results)

print("\n===== 전체 결과 =====")
print(df_result.to_string(index=False))

print("\n===== 분포 (describe) =====")
print(df_result.drop(columns='주소').describe().round(4).to_string())

print("\n===== 상위 10% (90th percentile) → MAX값 후보 =====")
cols = ['s_elem_dist', 'n_elem', 's_middle_dist', 'n_middle', 'n_academy', 'n_daycare', 'n_afterschool']
for col in cols:
    p90 = df_result[col].quantile(0.9)
    print(f"  {col}: {p90:.4f}")

완료: 서울특별시 강남구 테헤란로 152
완료: 서울특별시 도봉구 도봉로110마길 42
완료: 서울특별시 은평구 통일로 684
완료: 서울특별시 중랑구 망우로 353
완료: 서울특별시 강서구 화곡로 302
완료: 경기도 성남시 분당구 정자일로 95
완료: 경기도 고양시 일산동구 중앙로 1256
완료: 경기도 용인시 기흥구 중부대로 579
완료: 인천광역시 연수구 인천타워대로 323
완료: 인천광역시 부평구 부평대로 168
완료: 경기도 하남시 대청로 10

===== 전체 결과 =====
                   주소  s_elem_dist  n_elem  s_middle_dist  n_middle  n_academy  n_daycare  n_afterschool
   서울특별시 강남구 테헤란로 152       0.0012       1         0.0051         6         52          7              1
서울특별시 도봉구 도봉로110마길 42       0.0042       3         0.0090         8         42         11              3
    서울특별시 은평구 통일로 684       0.0086       5         0.0000         0         37         11              5
    서울특별시 중랑구 망우로 353       0.0067       5         0.0090         7         32          6              5
    서울특별시 강서구 화곡로 302       0.0083       5         0.0065         7         21          8              4
  경기도 성남시 분당구 정자일로 95       0.0161       3         0.0124         6         74          5     

In [22]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

def count_within(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    return int((df['거리'] <= radius).sum())

def normalize(value, v_min, v_max):
    value = max(v_min, min(v_max, value))
    return round((value - v_min) / (v_max - v_min), 4)

# ============================================================
# 반경 기준
# ============================================================
# 초등학교 1000m : 교육환경 보호에 관한 법률 - 초등학교 통학거리 법적 기준
# 중학교   1500m : 초등학교 법적 기준 준용, 중학생 보행 능력 고려 상향
# 학원      500m : 어린이보호구역 지정 기준 - 학교 정문 반경 500m
# 어린이집  500m : 분포 확인 후 조정
# 돌봄교실 1000m : 초등학교 내 운영이므로 초등학교 통학거리 기준 동일 적용

ELEM_RADIUS        = 1000
MIDDLE_RADIUS      = 1500
ACADEMY_RADIUS     = 500
DAYCARE_RADIUS     = 500
AFTERSCHOOL_RADIUS = 1000

# ============================================================
# 정규화 기준값 - 수도권 샘플 지역 90th percentile 기준
# ============================================================
ELEM_DIST_MAX   = 0.0115
ELEM_CNT_MAX    = 5
MIDDLE_DIST_MAX = 0.0090
MIDDLE_CNT_MAX  = 7
ACADEMY_MAX     = 52
DAYCARE_MAX     = 11
AFTERSCHOOL_MAX = 5

# ============================================================
# 데이터 로드
# ============================================================
df_elem        = pd.read_csv('교육 - 초등학교.csv',         encoding='cp949')
df_middle      = pd.read_csv('교육 - 중학교.csv',           encoding='cp949')
df_academy     = pd.read_csv('교육 - 학원.csv',             encoding='cp949')
df_daycare     = pd.read_csv('어린이집정제.csv', encoding='cp949')
df_afterschool = pd.read_csv('최종-방과후돌봄교실.csv',      encoding='cp949')

# 위도/경도 숫자로 변환
for df in [df_elem, df_middle, df_daycare, df_afterschool]:
    df['위도'] = pd.to_numeric(df['위도'], errors='coerce')
    df['경도'] = pd.to_numeric(df['경도'], errors='coerce')

df_academy['위도'] = pd.to_numeric(df_academy['위도'], errors='coerce')
df_academy['경도'] = pd.to_numeric(df_academy['경도'], errors='coerce')

# ============================================================
# 주소 입력
# ============================================================
address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
    # 점수 계산
    s_elem_dist   = get_access_score(my_lat, my_lng, df_elem,        '위도', '경도', ELEM_RADIUS)
    n_elem        = count_within    (my_lat, my_lng, df_elem,        '위도', '경도', ELEM_RADIUS)
    s_middle_dist = get_access_score(my_lat, my_lng, df_middle,      '위도', '경도', MIDDLE_RADIUS)
    n_middle      = count_within    (my_lat, my_lng, df_middle,      '위도', '경도', MIDDLE_RADIUS)
    n_academy     = count_within    (my_lat, my_lng, df_academy,     '위도', '경도', ACADEMY_RADIUS)
    n_daycare     = count_within    (my_lat, my_lng, df_daycare,     '위도', '경도', DAYCARE_RADIUS)
    n_afterschool = count_within    (my_lat, my_lng, df_afterschool, '위도', '경도', AFTERSCHOOL_RADIUS)

    # 정규화 (0~1)
    v1 = normalize(s_elem_dist,   0, ELEM_DIST_MAX)
    v2 = normalize(n_elem,        0, ELEM_CNT_MAX)
    v3 = normalize(s_middle_dist, 0, MIDDLE_DIST_MAX)
    v4 = normalize(n_middle,      0, MIDDLE_CNT_MAX)
    v5 = normalize(n_academy,     0, ACADEMY_MAX)
    v6 = normalize(n_daycare,     0, DAYCARE_MAX)
    v7 = normalize(n_afterschool, 0, AFTERSCHOOL_MAX)

    # 100점 만점 환산
    score = round((v1 + v2 + v3 + v4 + v5 + v6 + v7) / 7 * 100, 2)

    # 결과 출력
    print(f"\n📍 {address}")
    print(f"   좌표: ({my_lat}, {my_lng})")
    print()
    print(f"초등학교 접근성 ({ELEM_RADIUS}m):        {s_elem_dist:.4f}  → V1: {v1:.4f}")
    print(f"초등학교 개수 ({ELEM_RADIUS}m):          {n_elem}개  → V2: {v2:.4f}")
    print(f"중학교 접근성 ({MIDDLE_RADIUS}m):       {s_middle_dist:.4f}  → V3: {v3:.4f}")
    print(f"중학교 개수 ({MIDDLE_RADIUS}m):         {n_middle}개  → V4: {v4:.4f}")
    print(f"학원 밀집도 ({ACADEMY_RADIUS}m):         {n_academy}개  → V5: {v5:.4f}")
    print(f"어린이집 개수 ({DAYCARE_RADIUS}m):       {n_daycare}개  → V6: {v6:.4f}")
    print(f"돌봄교실 운영 학교 ({AFTERSCHOOL_RADIUS}m):  {n_afterschool}개  → V7: {v7:.4f}")
    print()
    print(f"교육점수 = (V1+V2+V3+V4+V5+V6+V7)/7 × 100")
    print(f"         = {score}점")

주소를 입력하세요:  서울시 도봉구 도봉로110마길 42



📍 서울시 도봉구 도봉로110마길 42
   좌표: (37.6445244741978, 127.036204968923)

초등학교 접근성 (1000m):        0.0042  → V1: 0.3676
초등학교 개수 (1000m):          3개  → V2: 0.6000
중학교 접근성 (1500m):       0.0090  → V3: 1.0000
중학교 개수 (1500m):         8개  → V4: 1.0000
학원 밀집도 (500m):         42개  → V5: 0.8077
어린이집 개수 (500m):       11개  → V6: 1.0000
돌봄교실 운영 학교 (1000m):  3개  → V7: 0.6000

교육점수 = (V1+V2+V3+V4+V5+V6+V7)/7 × 100
         = 76.79점


In [23]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

def count_within(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    return int((df['거리'] <= radius).sum())

def normalize(value, v_min, v_max):
    value = max(v_min, min(v_max, value))
    return round((value - v_min) / (v_max - v_min), 4)

# ============================================================
# 반경 기준
# ============================================================
# 초등학교 1000m : 교육환경 보호에 관한 법률 - 초등학교 통학거리 법적 기준
# 중학교   1500m : 초등학교 법적 기준 준용, 중학생 보행 능력 고려 상향
# 학원      500m : 어린이보호구역 지정 기준 - 학교 정문 반경 500m
# 어린이집  500m : 분포 확인 후 조정
# 돌봄교실 1000m : 초등학교 내 운영이므로 초등학교 통학거리 기준 동일 적용

ELEM_RADIUS        = 1000
MIDDLE_RADIUS      = 1500
ACADEMY_RADIUS     = 500
DAYCARE_RADIUS     = 500
AFTERSCHOOL_RADIUS = 1000

# ============================================================
# 정규화 기준값 - 수도권 샘플 지역 90th percentile 기준
# ============================================================
ELEM_DIST_MAX   = 0.0115
ELEM_CNT_MAX    = 5
MIDDLE_DIST_MAX = 0.0090
MIDDLE_CNT_MAX  = 7
ACADEMY_MAX     = 52
DAYCARE_MAX     = 11
AFTERSCHOOL_MAX = 5

# ============================================================
# 데이터 로드
# ============================================================
df_elem        = pd.read_csv('교육 - 초등학교.csv',         encoding='cp949')
df_middle      = pd.read_csv('교육 - 중학교.csv',           encoding='cp949')
df_academy     = pd.read_csv('교육 - 학원.csv',             encoding='cp949')
df_daycare     = pd.read_csv('어린이집정제.csv', encoding='cp949')
df_afterschool = pd.read_csv('최종-방과후돌봄교실.csv',      encoding='cp949')

# 위도/경도 숫자로 변환
for df in [df_elem, df_middle, df_daycare, df_afterschool]:
    df['위도'] = pd.to_numeric(df['위도'], errors='coerce')
    df['경도'] = pd.to_numeric(df['경도'], errors='coerce')

df_academy['위도'] = pd.to_numeric(df_academy['위도'], errors='coerce')
df_academy['경도'] = pd.to_numeric(df_academy['경도'], errors='coerce')

# ============================================================
# 주소 입력
# ============================================================
address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
    # 점수 계산
    s_elem_dist   = get_access_score(my_lat, my_lng, df_elem,        '위도', '경도', ELEM_RADIUS)
    n_elem        = count_within    (my_lat, my_lng, df_elem,        '위도', '경도', ELEM_RADIUS)
    s_middle_dist = get_access_score(my_lat, my_lng, df_middle,      '위도', '경도', MIDDLE_RADIUS)
    n_middle      = count_within    (my_lat, my_lng, df_middle,      '위도', '경도', MIDDLE_RADIUS)
    n_academy     = count_within    (my_lat, my_lng, df_academy,     '위도', '경도', ACADEMY_RADIUS)
    n_daycare     = count_within    (my_lat, my_lng, df_daycare,     '위도', '경도', DAYCARE_RADIUS)
    n_afterschool = count_within    (my_lat, my_lng, df_afterschool, '위도', '경도', AFTERSCHOOL_RADIUS)

    # 정규화 (0~1)
    v1 = normalize(s_elem_dist,   0, ELEM_DIST_MAX)
    v2 = normalize(n_elem,        0, ELEM_CNT_MAX)
    v3 = normalize(s_middle_dist, 0, MIDDLE_DIST_MAX)
    v4 = normalize(n_middle,      0, MIDDLE_CNT_MAX)
    v5 = normalize(n_academy,     0, ACADEMY_MAX)
    v6 = normalize(n_daycare,     0, DAYCARE_MAX)
    v7 = normalize(n_afterschool, 0, AFTERSCHOOL_MAX)

    # 100점 만점 환산
    score = round((v1 + v2 + v3 + v4 + v5 + v6 + v7) / 7 * 100, 2)

    # 결과 출력
    print(f"\n📍 {address}")
    print(f"   좌표: ({my_lat}, {my_lng})")
    print()
    print(f"초등학교 접근성 ({ELEM_RADIUS}m):        {s_elem_dist:.4f}  → V1: {v1:.4f}")
    print(f"초등학교 개수 ({ELEM_RADIUS}m):          {n_elem}개  → V2: {v2:.4f}")
    print(f"중학교 접근성 ({MIDDLE_RADIUS}m):       {s_middle_dist:.4f}  → V3: {v3:.4f}")
    print(f"중학교 개수 ({MIDDLE_RADIUS}m):         {n_middle}개  → V4: {v4:.4f}")
    print(f"학원 밀집도 ({ACADEMY_RADIUS}m):         {n_academy}개  → V5: {v5:.4f}")
    print(f"어린이집 개수 ({DAYCARE_RADIUS}m):       {n_daycare}개  → V6: {v6:.4f}")
    print(f"돌봄교실 운영 학교 ({AFTERSCHOOL_RADIUS}m):  {n_afterschool}개  → V7: {v7:.4f}")
    print()
    print(f"교육점수 = (V1+V2+V3+V4+V5+V6+V7)/7 × 100")
    print(f"         = {score}점")

주소를 입력하세요:  서울 중구 퇴계로 166 아시아경제교육센터 3층



📍 서울 중구 퇴계로 166 아시아경제교육센터 3층
   좌표: (37.5608939235742, 126.990166216967)

초등학교 접근성 (1000m):        0.0091  → V1: 0.7898
초등학교 개수 (1000m):          4개  → V2: 0.8000
중학교 접근성 (1500m):       0.0013  → V3: 0.1395
중학교 개수 (1500m):         1개  → V4: 0.1429
학원 밀집도 (500m):         9개  → V5: 0.1731
어린이집 개수 (500m):       2개  → V6: 0.1818
돌봄교실 운영 학교 (1000m):  3개  → V7: 0.6000

교육점수 = (V1+V2+V3+V4+V5+V6+V7)/7 × 100
         = 40.39점
